# SENTINEL — Thử nghiệm luồng A trên vùng người (person-group crops)

**Vì sao:** chẩn đoán `visual_fight_v1` cho thấy S3D nhìn *toàn khung hình* thu về 224 px không thấy đánh nhau trên camera giám sát: zero-shot Kinetics trong video 0,513; bỏ bối cảnh thì AUC theo frame rơi 0,82 → 0,52; không cách nào vượt đường cơ sở chuyển động trong video (0,562).

**Thử nghiệm này:** chạy S3D trên vùng cắt vuông quanh tối đa 3 nhóm người lớn nhất của mỗi giây (khung người lấy từ khung xương đã trích của luồng B, `features/pose_v1`), đặc trưng của clip = max theo các vùng cắt. Chỉ trích **video UBI-Fights có đánh nhau** (đủ để đo AUC trong video, ít hơn nhiều so với toàn bộ 80 giờ) và Surveillance Camera Fight, rồi chạy lại đúng bảng chẩn đoán.

**Đọc kết quả (mục 7, dòng `KẾT LUẬN` cũng ghi vào `artifacts/visual_crops_pilot/run_log.txt`):** so cột *test trong video* với toàn khung hình (tốt nhất 0,589) và đường cơ sở chuyển động (0,562).
- Vượt rõ (`SIGNAL`) → trích vùng người cho toàn bộ dataset và train lại luồng A trên đó.
- Không vượt → cả khung xương lẫn hình ảnh vùng người đều không tách được đoạn đánh với đoạn yên trên UBI-Fights; cần xem lại nhãn / thước đo trước khi đầu tư thêm.

AUC theo frame của lần này chỉ tính trên video có đánh nhau nên **không** so được với lần trước; chỉ so AUC trong video và SCF.

**GPU:** L4 (giải mã video là nút thắt). Khoảng 45–90 phút. `Runtime > Run all`, giữ tab mở; chạy xong hoặc lỗi thì tự ngắt runtime.

## 0. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
ZIP_DIR = DRIVE_ROOT / "datasets" / "fight"            # UBI_FIGHTS.zip của các lần trước
POSE_ROOT = DRIVE_ROOT / "features" / "pose_v1"         # khung người từ khung xương của luồng B
FEATURE_ROOT = DRIVE_ROOT / "features" / "s3d_crops_v1" # đặc trưng vùng người (mới)
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "visual_crops_pilot"
WHOLE_FRAME_DIAGNOSTICS = DRIVE_ROOT / "artifacts" / "visual_fight_v1" / "diagnostics" / "visual_diagnostics.json"
DEDUP_DIR = DRIVE_ROOT / "artifacts" / "fight" / "dedup"
LOCAL_DATA = Path("/content/datasets")
UBI_ROOT = LOCAL_DATA / "UBI_FIGHTS"
SCF_ROOT = LOCAL_DATA / "fight-detection-surv-dataset"
SCF_URL = "https://github.com/sayibet/fight-detection-surv-dataset.git"
EXTRACT_BATCH = 16             # số vùng cắt mỗi lần chạy S3D
AUTO_DISCONNECT = True
print("Features:", FEATURE_ROOT, "| Kết quả:", ARTIFACT_DIR)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [REPO_DIR.parent, ZIP_DIR, FEATURE_ROOT, ARTIFACT_DIR, LOCAL_DATA]:
    path.mkdir(parents=True, exist_ok=True)

import time
from contextlib import contextmanager
RUN_LOG = ARTIFACT_DIR / "run_log.txt"

def note(message):
    # Ghi một dòng vào log trên Drive (không ngắt runtime).
    with open(RUN_LOG, "a", encoding="utf-8") as log:
        log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")
    print(message)

def finish_and_disconnect(message):
    # Ghi log lên Drive; khi AUTO_DISCONNECT thì đồng bộ Drive rồi ngắt runtime để dừng tốn compute unit.
    note(message)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

@contextmanager
def step(name):
    try:
        yield
    except Exception as error:
        finish_and_disconnect(f"LỖI ở mục {name}: {error!r} — xem output của mục này, sửa rồi chạy lại.")
        raise

print("Log chạy:", RUN_LOG, "| AUTO_DISCONNECT =", AUTO_DISCONNECT)

## 2. Đồng bộ code mới nhất

In [ ]:
with step('2. Đồng bộ code'):
    import os, subprocess

    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Cài dependency, tải trọng số S3D

In [ ]:
with step('3. Cài dependency'):
    import sys, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CPUS = os.cpu_count() or 2
    EXTRACT_SHARDS = max(1, min(8, CPUS - 1))   # tiến trình trích đặc trưng song song (giải mã video là nút thắt)
    print("PyTorch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHÔNG CÓ GPU")
    print(f"CPU: {CPUS} | extract shards: {EXTRACT_SHARDS}")
    from torchvision.models.video import S3D_Weights
    S3D_Weights.KINETICS400_V1.get_state_dict(progress=False)
    print("Trọng số S3D:", S3D_Weights.KINETICS400_V1.url)

## 4. Video đánh nhau của UBI-Fights + nhãn
Từ `UBI_FIGHTS.zip` trên Drive chỉ giải nén **video đánh nhau**, file nhãn và `test_videos.csv`; video bình thường được thay bằng file rỗng cùng tên (chỉ cần cho cách chia train/validation). Surveillance Camera Fight clone từ GitHub.

In [ ]:
with step('4. Dữ liệu'):
    import zipfile
    ubi_zip = ZIP_DIR / "UBI_FIGHTS.zip"
    assert ubi_zip.exists(), f"Không thấy {ubi_zip}"
    assert any(POSE_ROOT.glob("ubi/*.npz")), f"Không có khung xương trong {POSE_ROOT}"
    with zipfile.ZipFile(ubi_zip) as archive:
        for info in archive.infolist():
            name = info.filename
            if name.endswith("/"):
                continue
            target = LOCAL_DATA / name
            if name.lower().endswith(".mp4") and "/fight/" not in name:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.touch()                                   # video bình thường: chỉ cần tên
            elif not target.exists() or target.stat().st_size != info.file_size:
                archive.extract(info, LOCAL_DATA)                # video đánh nhau, nhãn, test_videos.csv
    print("UBI-Fights video đánh nhau:", len(list(UBI_ROOT.glob("videos/fight/*.mp4"))),
          "| file nhãn:", len(list(UBI_ROOT.glob("annotation/*.csv"))))
    if not SCF_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", SCF_URL, str(SCF_ROOT)], check=True)
    print("Surveillance Camera Fight:", len(list(SCF_ROOT.rglob("*.mp4"))), "clip")

    def run(script, *args):
        subprocess.run([sys.executable, "-u", script, *[str(a) for a in args]], cwd=REPO_DIR, check=True)

## 5. Smoke test
Trích vùng người cho 2 video đánh nhau của tập test vào thư mục tạm, xem số vùng cắt mỗi giây và tốc độ.

In [ ]:
with step('5. Smoke test'):
    import json
    import numpy as np
    smoke_root = Path("/content/crops_smoke")
    run("scripts/extract_visual.py", "--dataset", f"ubi:{UBI_ROOT}", "--feature-root", smoke_root, "--crops",
        "--pose-root", POSE_ROOT, "--fight-only", "--split", "test", "--limit", "2", "--device", DEVICE,
        "--batch-size", EXTRACT_BATCH, "--force")
    videos = json.loads((smoke_root / "ubi" / "manifest.json").read_text())["videos"]
    for key, info in videos.items():
        with np.load(smoke_root / "ubi" / f"{key}.npz") as data:
            counts = data["crop_count"]
            assert data["features"].shape[1] == 2048 and np.isfinite(data["features"].astype(np.float32)).all()
        print(f"{key}: {info['clips']} clip, {info['seconds']} s, vùng cắt/clip trung bình {counts.mean():.2f}")

## 6. Trích đặc trưng vùng người (song song)
Giai đoạn 1: video UBI-Fights có đánh nhau (`--fight-only`). Giai đoạn 2: toàn bộ Surveillance Camera Fight (cần cả clip không đánh nhau để đo AUC theo clip). Mỗi video một file trên Drive; chạy lại cell sau khi Colab ngắt thì video đã có được bỏ qua.

In [ ]:
with step('6. Trích đặc trưng vùng người'):
    import time

    def extract_parallel(spec, extra):
        processes = []
        for shard in range(EXTRACT_SHARDS):
            log = open(f"/content/crops_shard{shard}.log", "w")
            command = [sys.executable, "-u", "scripts/extract_visual.py", "--dataset", spec,
                       "--feature-root", str(FEATURE_ROOT), "--crops", "--pose-root", str(POSE_ROOT),
                       "--device", DEVICE, "--batch-size", str(EXTRACT_BATCH),
                       "--shard", f"{shard}/{EXTRACT_SHARDS}", *extra]
            processes.append((subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT), log))
        started = time.time()
        while any(process.poll() is None for process, _ in processes):
            time.sleep(120)
            print(f"[{(time.time() - started) / 60:.0f} phút]")
            for shard in range(EXTRACT_SHARDS):
                lines = Path(f"/content/crops_shard{shard}.log").read_text(errors="ignore").strip().splitlines()
                print(f"   shard {shard}: {(lines[-1] if lines else '(chưa có output)')[:110]}")
        for shard, (process, log) in enumerate(processes):
            log.close()
            if process.returncode != 0:
                print(Path(f"/content/crops_shard{shard}.log").read_text(errors="ignore")[-3000:])
                raise RuntimeError(f"Shard {shard} lỗi (mã {process.returncode}); chạy lại cell để tiếp tục")

    extract_parallel(f"ubi:{UBI_ROOT}", ["--fight-only"])
    extract_parallel(f"scf:{SCF_ROOT}", [])
    print("Xong:", sum(1 for _ in FEATURE_ROOT.glob("ubi/*.npz")), "video UBI,",
          sum(1 for _ in FEATURE_ROOT.glob("scf/*.npz")), "clip SCF")

## 7. Chẩn đoán trên vùng người, so với toàn khung hình

In [ ]:
with step('7. Chẩn đoán'):
    run("scripts/diagnose_visual_fight.py", "--dataset", f"ubi:{UBI_ROOT}", "--dataset", f"scf:{SCF_ROOT}",
        "--feature-root", FEATURE_ROOT, "--output-dir", ARTIFACT_DIR / "diagnostics",
        "--exclude", DEDUP_DIR / "exclude.json", "--device", DEVICE)
    crops = json.loads((ARTIFACT_DIR / "diagnostics" / "visual_diagnostics.json").read_text())
    whole = json.loads(WHOLE_FRAME_DIAGNOSTICS.read_text())["results"] if WHOLE_FRAME_DIAGNOSTICS.exists() else {}
    header = f"{'cách':20s} {'trong video (toàn khung)':>25s} {'trong video (vùng người)':>25s} {'SCF (toàn khung)':>17s} {'SCF (vùng người)':>17s}"
    print(header); print("-" * len(header))
    fmt = lambda value: "" if value is None else f"{value:.3f}"
    for name, row in crops["results"].items():
        before = whole.get(name, {})
        print(f"{name:20s} {fmt(before.get('test_video_auc_median')):>25s} {fmt(row['test_video_auc_median']):>25s} "
              f"{fmt(before.get('scf_clip_auc')):>17s} {fmt(row.get('scf_clip_auc')):>17s}")
    note(f"KẾT LUẬN thử nghiệm vùng người: {crops['verdict']}")

## 8. Kết thúc: ngắt runtime

In [ ]:
finish_and_disconnect(f"Hoàn tất thử nghiệm vùng người. Kết quả: {ARTIFACT_DIR / 'diagnostics' / 'visual_diagnostics.json'}")